In [1]:
import math
from random import randint
import torch
 
from torch import nn
 
 
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.d_k = d_model // n_head
        self.n_head = n_head
        self.linears = nn.ModuleList(
            [nn.Linear(d_model, d_model) for _ in range(4)]
        )
 
    def forward(self, x, mask):
        '''
        x: (batch_size, seq_len, d_model)
        '''
        batch_size = x.size()[0]
        q, k, v = [l(x).view(
            batch_size, -1, self.n_head, self.d_k).transpose(1, 2) for l in self.linears[:3]]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsequeeze(1)
            if mask.dim() == 2:
                mask = mask[:, None, None, :]
            scores = scores.masked_fill(mask == 0, -1e9)
 
        scores = torch.softmax(scores, dim=-1)
        concat_multi_head_seq = torch.matmul(scores, v).transpose(
            1, 2).contiguous().view(
            batch_size, -1, self.n_head * self.d_k)
        context_seq = self.linears[-1](concat_multi_head_seq)
        return context_seq, scores

In [2]:
max_seq_len = 5
num_head = 2
d_model = 4
vocab_size = 100
batch_size = 2
 
att = MultiHeadAttention(d_model, num_head)
# 造出batch_size个句子的序列embedding
x = torch.randn((batch_size, max_seq_len, d_model))
# 造出句子的token长度
sentences_lens = [randint(1, max_seq_len) for _ in range(batch_size)]
# 造出句子的mask
mask = torch.tensor([
    [1 if i <= per_sentence_len else 0 for i in range(max_seq_len)]
    for per_sentence_len in sentences_lens])
 
attention_context, attention_scores = att(x, mask)
print(attention_context.shape)
print(attention_scores.shape)

torch.Size([2, 5, 4])
torch.Size([2, 2, 5, 5])


In [4]:
masked_att = MultiHeadAttention(d_model, num_head)
 
# 造出batch_size个句子的序列embedding
x1 = torch.randn((batch_size, max_seq_len, d_model))
# 造出句子的token长度
sentences_lens1 = [randint(1, max_seq_len) for _ in range(batch_size)]
# 造出句子的mask
pad_mask = torch.tensor([
    [1 if i <= per_sentence_len else 0 for i in range(max_seq_len)]
    for per_sentence_len in sentences_lens1])
 
# 生成一个下三角矩阵作为掩码
triangular_mask = torch.tril(
    torch.ones((max_seq_len, max_seq_len)).type_as(pad_mask))


In [6]:
sentences_lens1

[1, 5]

In [ ]:

# pad_mask位与casual_mask
mask1 = pad_mask.unsqueeze(1) & triangular_mask.unsqueeze(0)
 
attention_context1, attention_scores1 = masked_att(x1, mask1)

In [15]:
seq_len = 3
torch.triu(
                torch.full((seq_len, seq_len), float("-inf")),
                diagonal=1
)

tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])

In [10]:
torch.full((seq_len, seq_len), float("-inf"))

tensor([[-inf, -inf, -inf],
        [-inf, -inf, -inf],
        [-inf, -inf, -inf]])

In [1]:
import torch
import math

T = torch.tensor([3])
P = torch.tensor([0.1, 0.2, 0.15, 0.25, 0.3])
# T中表示的是用户的需求项，3表示第3项。
# P中表示的是概率值，即为对应index项被推荐的概率。
# 如0.1的index为0，表示第0项被推荐的概率为0.1
# T和P中仅包含一次推荐过程：用户需要的第3个项目在推荐列表中排在第2位。
# 第2位是指将P按概率值对index进行排位的结果(4, 3, 1, 2, 0)的第二个元素。


def MRR(truth, pred, N):
    """
    Mean Reciprocal Rank
    truth: the index of real target (from 0 to len(truth)-1)
    pred: rank list
    N: top-N elements of rank list
    """
    top_N = pred.squeeze()[:N]
    if truth not in top_N:
        rr_N = 0.0
    else:
        i = top_N.numpy().tolist().index(truth)
        rr_N = 1 / (i + 1) # index从0开始
    return rr_N


p = torch.argsort(P.unsqueeze(0), dim=1, descending=True)
mrr = MRR(T, p, 3)
print(f'MRR={mrr}\n')

MRR=0.5

